In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone -b m3-1-josh https://github.com/jkelle11-source/msml612_project.git
%cd msml612_project
!uv sync --locked
!uv run python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!mkdir -p data/processed && cp /content/drive/MyDrive/msml612/physionet_set_a.npz data/processed/
!grep -c use_deterministic_kernels diffusion/train.py

Cloning into 'msml612_project'...
remote: Enumerating objects: 333, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 333 (delta 34), reused 34 (delta 32), pack-reused 258 (from 1)
Receiving objects: 100% (333/333), 316.10 KiB | 7.35 MiB/s, done.
Resolving deltas: 100% (170/170), done.
/content/msml612_project
Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 67 packages in 32ms
Prepared 63 packages in 49.85s
Installed 63 packages in 551ms
 + annotated-types==0.8.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + einops==0.8.2
 + filelock==4.0.12
 + fsspec==2026.9.0
 + googleapis-common-protos==1.75.5
 + idna==3.20
 + iniconfig==2.3.1
 + jinja2==3.1.6
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.7
 + numpy==2.5.3
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0

In [3]:
!uv run python -m diffusion.run --seed 42 --device cuda --epochs 2 --out /content/det_a
!uv run python -m diffusion.run --seed 42 --device cuda --epochs 2 --out /content/det_b

base seed 42 on cuda, 2 epochs -> /content/det_a
/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)
epoch 1/2  loss 0.3985  max grad norm 7.56  [32s]
epoch 2/2  loss 0.3420  max grad norm 1.70  val MAE 0.5550  val RMSE 1.1022  [val 286s]  (forward fill 0.4366)  [26s]
{
 "variant": "base",
 "seed": 42,
 "device": "cuda",
 "validation": {
  "split": "val",
  "n_samples": 5,
  "batch_size": 64
 },
 "health": {
  "n_steps": 350,
  "finite": true,
  "loss_first_tenth": 0.5559472437415804,
  "loss_last_tenth": 0.34168134672301154,
  "grad_norm_median": 0.657866358757019,
  "grad_norm_max": 7.555301189422607,
  "grad_norm_max_step": 16,
  "spikes": [
   {
    "step": 16,
    "loss": 0.5120609402656555,
    "grad_norm": 7.555301189422607
   }

In [4]:
%%writefile /content/compare_runs.py
"""Compare two diffusion.run output folders bit for bit: python compare_runs.py <run a> <run b>."""
import json
import sys
from pathlib import Path

import torch

a, b = Path(sys.argv[1]), Path(sys.argv[2])
strip = lambda rows: [{k: v for k, v in r.items() if not k.endswith("seconds")} for r in rows]
history = [(p / "history.jsonl").read_text().splitlines() for p in (a, b)]
curve = [json.loads((p / "curve.json").read_text()) for p in (a, b)]
summary = [json.loads((p / "summary.json").read_text()) for p in (a, b)]
weights = [torch.load(p / "model.pt", map_location="cpu") for p in (a, b)]
same_weights = weights[0].keys() == weights[1].keys() and all(torch.equal(weights[0][k], weights[1][k]) for k in weights[0])
first_diff = next((i for i, (x, y) in enumerate(zip(*history)) if x != y), None)

print(f"history:  {len(history[0])} vs {len(history[1])} steps, identical: {history[0] == history[1]}"
      + ("" if first_diff is None else f" (first difference at step {first_diff})"))
print(f"curve:    identical apart from timings: {strip(curve[0]) == strip(curve[1])}")
print(f"summary:  identical: {summary[0] == summary[1]}")
print(f"model.pt: bit-identical: {same_weights}")
print("seconds per epoch:", [[r["seconds"] for r in c] for c in curve])


Writing /content/compare_runs.py


In [5]:
!uv run python /content/compare_runs.py /content/det_a /content/det_b

history:  350 vs 350 steps, identical: True
curve:    identical apart from timings: True
summary:  identical: True
model.pt: bit-identical: True
seconds per epoch: [[32.3, 25.7], [28.9, 26.7]]
